# Traffic Demand Prediction - Advanced Ensemble Approach

This notebook builds upon previous optimizations and introduces state-of-the-art time-series techniques:
1. **Exact 24-Hour Lag Features:** Matches exact location and time from Day 48 to predict Day 49.
2. **Rolling Window Averages:** Captures the traffic trend (building up or dying down) around the target time.
3. **Log-Transform Target Variable:** Uses `np.log1p()` to compress extreme traffic spikes, allowing trees to optimize effectively.
4. **LightGBM + XGBoost Ensembling:** Blends the predictions of two powerful gradient boosting frameworks for a smoother, more robust result.

In [ ]:
# Install required dependencies
!pip install pygeohash xgboost lightgbm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import pygeohash as pgh
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

### 1. Load Datasets

In [ ]:
train_url = "https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/train.csv"
test_url = "https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/test.csv"

train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

### 2. Basic Preprocessing & Missing Values

In [ ]:
def preprocess_basics(df):
    df = df.copy()
    df["RoadType"] = df["RoadType"].fillna("Unknown")
    df["Weather"] = df["Weather"].fillna("Unknown")
    df["LargeVehicles"] = df["LargeVehicles"].replace({"Allowed": 1, "Not Allowed": 0}).fillna(0)
    df["Landmarks"] = df["Landmarks"].replace({"Yes": 1, "No": 0}).fillna(0)
    df["Temperature"] = df.groupby("Weather")["Temperature"].transform(lambda x: x.fillna(x.mean()))
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())
    return df

train = preprocess_basics(train)
test = preprocess_basics(test)

### 3. Geohash Decoding & Time Features

In [ ]:
def add_spatial_time_features(df):
    df = df.copy()
    # Spatial
    df["latitude"] = df["geohash"].apply(lambda x: pgh.decode(x)[0])
    df["longitude"] = df["geohash"].apply(lambda x: pgh.decode(x)[1])
    
    # Time
    df[["hour", "minute"]] = df["timestamp"].str.split(":", expand=True).astype(int)
    df["time_in_mins"] = df["hour"] * 60 + df["minute"]
    df["sin_time"] = np.sin(2 * np.pi * df["time_in_mins"] / 1440)
    df["cos_time"] = np.cos(2 * np.pi * df["time_in_mins"] / 1440)
    
    def get_part_of_day(hour):
        if 5 <= hour < 12: return "Morning"
        elif 12 <= hour < 17: return "Afternoon"
        elif 17 <= hour < 21: return "Evening"
        else: return "Night"
    
    df["part_of_day"] = df["hour"].apply(get_part_of_day)
    df.drop(["timestamp"], axis=1, inplace=True)
    return df

train = add_spatial_time_features(train)
test = add_spatial_time_features(test)

### 4. Advanced Lag & Trend Features (24-Hour Lookback)

In [ ]:
# 1. Calculate Daily Historical Average for Day 48 (Fallback)
historical_avg = train[train["day"] == 48].groupby("geohash")["demand"].mean().reset_index()
historical_avg.rename(columns={"demand": "hist_avg_demand"}, inplace=True)

# 2. Calculate Exact 24-hour Lag and Rolling Window Average
day48 = train[train["day"] == 48].copy()
day48 = day48.sort_values(["geohash", "time_in_mins"])
day48["lag_rolling_avg"] = day48.groupby("geohash")["demand"].transform(
    lambda x: x.rolling(window=3, min_periods=1, center=True).mean()
)

lag_features = day48[["geohash", "time_in_mins", "demand", "lag_rolling_avg"]]
lag_features.rename(columns={"demand": "lag_24h_demand"}, inplace=True)

# 3. Merge Historical Averages into Train and Test
train = train.merge(historical_avg, on="geohash", how="left")
test = test.merge(historical_avg, on="geohash", how="left")

# Unseen geohashes fallback to global mean
global_mean_day48 = train[train["day"] == 48]["demand"].mean()
train["hist_avg_demand"] = train["hist_avg_demand"].fillna(global_mean_day48)
test["hist_avg_demand"] = test["hist_avg_demand"].fillna(global_mean_day48)

# 4. Merge Exact Lag Features into Train and Test
train = train.merge(lag_features, on=["geohash", "time_in_mins"], how="left")
test = test.merge(lag_features, on=["geohash", "time_in_mins"], how="left")

# 5. Fallback logic: If specific lag time is missing, fallback to location's daily average
train["lag_24h_demand"] = train["lag_24h_demand"].fillna(train["hist_avg_demand"])
train["lag_rolling_avg"] = train["lag_rolling_avg"].fillna(train["hist_avg_demand"])

test["lag_24h_demand"] = test["lag_24h_demand"].fillna(test["hist_avg_demand"])
test["lag_rolling_avg"] = test["lag_rolling_avg"].fillna(test["hist_avg_demand"])

# Safely drop geohash string now
train.drop(["geohash"], axis=1, inplace=True)
test.drop(["geohash"], axis=1, inplace=True)

### 5. Encoding Categorical Columns

In [ ]:
categorical_cols = ["RoadType", "Weather", "part_of_day"]
train = pd.get_dummies(train, columns=categorical_cols)
test = pd.get_dummies(test, columns=categorical_cols)

test = test.reindex(columns=[col for col in train.columns if col != "demand"], fill_value=0)

### 6. Time-based Splitting & Target Transformation

In [ ]:
features = train.drop(["demand", "Index"], axis=1)
target = train["demand"]

# Train on Day 48
X_train = features[train["day"] == 48]
y_train = target[train["day"] == 48]

# Validate on Day 49
X_val = features[train["day"] == 49]
y_val = target[train["day"] == 49]

# LOG TRANSFORMATION OF TARGET VARIABLE (Crucial for skewness)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

### 7. Model Training (XGBoost + LightGBM)

In [ ]:
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=1500, max_depth=6, learning_rate=0.03, 
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    early_stopping_rounds=50, objective="reg:squarederror"
)
xgb_model.fit(
    X_train, y_train_log, 
    eval_set=[(X_train, y_train_log), (X_val, y_val_log)], 
    verbose=200
)

print("\nTraining LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=1500, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
try:
    lgb_model.fit(
        X_train, y_train_log,
        eval_set=[(X_train, y_train_log), (X_val, y_val_log)],
        callbacks=[lgb.early_stopping(50, verbose=200)]
    )
except AttributeError:
    # Fallback for older LightGBM versions
    lgb_model.fit(
        X_train, y_train_log,
        eval_set=[(X_train, y_train_log), (X_val, y_val_log)],
        early_stopping_rounds=50, verbose=200
    )

### 8. Evaluation & Ensembling

In [ ]:
# Predict on Validation Set
xgb_val_preds_log = xgb_model.predict(X_val)
lgb_val_preds_log = lgb_model.predict(X_val)

# INVERSE TRANSFORMATION (expm1)
xgb_val_preds = np.expm1(xgb_val_preds_log)
lgb_val_preds = np.expm1(lgb_val_preds_log)

# Blend predictions (Simple Average)
y_pred_val = (xgb_val_preds + lgb_val_preds) / 2

val_r2 = r2_score(y_val, y_pred_val)
val_mse = mean_squared_error(y_val, y_pred_val)

print(f"Validation MSE: {val_mse:.5f}")
print(f"Validation R2 Score: {val_r2:.5f}")
print(f"\nEstimated Final Evaluation Score: {max(0, 100 * val_r2):.2f}")

### 9. Test Predictions & Submission

In [ ]:
test_features = test.drop(["Index"], axis=1)

# Predict & Inverse Transform
xgb_test_preds = np.expm1(xgb_model.predict(test_features))
lgb_test_preds = np.expm1(lgb_model.predict(test_features))

# Blend
final_test_preds = (xgb_test_preds + lgb_test_preds) / 2

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_test_preds
})

submission.to_csv("submissions/submission_advanced_ensemble.csv", index=False)

print("Submission file 'submission_advanced_ensemble.csv' successfully created!")
display(submission.head())